# Redes híbridas con PyTorch

`polypus.torch.QuantumLayer` convierte un modelo nativo en una capa de PyTorch, un `torch.nn.Module`, así que puede ir entre capas clásicas y entrenarse con cualquier optimizador de `torch.optim`. Su paso hacia delante es `qml.expval` y su paso hacia atrás usa `qml.vjp`, que multiplica de una vez el gradiente que llega por el `qml.jacobian` de [`07c`](07c_expval_y_jacobian.ipynb), sin construir el jacobiano entero: la diferenciación automática de PyTorch calcula los gradientes de las capas clásicas, y Polypus calcula el de la capa cuántica de forma exacta.

PyTorch no se instala con Polypus. Viene como extra opcional:

```bash
pip install "polypus-quantum[torch]"
```

`import polypus` no importa PyTorch; solo lo hace `polypus.torch`.

## La capa cuántica

El modelo tiene dos qubits y una lectura de dos observables, `⟨Z₀⟩` y `⟨Z₁⟩`, así que la capa recibe dos ángulos por muestra y devuelve dos números por muestra. Sus parámetros entrenables son los del modelo, en `params`, un `torch.nn.Parameter` sacado al azar del generador de PyTorch, así que `torch.manual_seed` los fija. `exact=True` usa valores esperados exactos, sin ruido de muestreo:

In [ ]:
import torch
from polypus import qml
from polypus.torch import QuantumLayer

torch.manual_seed(3)

modelo = qml.Model(
    qml.Encoder(2).angle_encoder("ry"),
    qml.Ansatz(2).real_amplitudes(1),
    qml.Readout([qml.Z(0), qml.Z(1)]),
)
cuantica = QuantumLayer(modelo, exact=True)
print(cuantica)
print(cuantica.params)

Al llamarla sobre un lote de muestras, la capa devuelve una fila por muestra y una columna por observable, un tensor que PyTorch puede derivar: `backward` rellena `cuantica.params.grad` con el gradiente exacto, que Polypus calcula por diferenciación adjunta porque la capa tiene `exact=True` (con shots lo estimaría con la regla parameter-shift). Después se borra el gradiente, para que el entrenamiento de abajo empiece sin gradientes acumulados:

In [ ]:
angulos = torch.tensor([[0.2, 0.1], [2.8, 3.0]])
salida = cuantica(angulos)
print(salida)

salida.sum().backward()
print(cuantica.params.grad)
cuantica.params.grad = None

## Una red híbrida

Ahora hay ocho muestras con cuatro características cada una, cuatro por clase. Una capa `Linear` lleva las cuatro características a los dos ángulos que recibe la capa cuántica, y una segunda `Linear` convierte sus dos salidas en un logit, la puntuación de la clase 1. Las capas clásicas trabajan en `float32`, el tipo por defecto de PyTorch; la capa cuántica convierte el tipo a la entrada y a la salida:

In [ ]:
X = torch.tensor(
    [
        [0.9, 0.8, 0.1, 0.0],
        [1.0, 0.6, 0.2, 0.1],
        [0.8, 0.9, 0.0, 0.2],
        [0.7, 1.0, 0.1, 0.1],
        [0.1, 0.2, 0.9, 1.0],
        [0.0, 0.1, 1.0, 0.8],
        [0.2, 0.0, 0.8, 0.9],
        [0.1, 0.1, 0.7, 1.0],
    ]
)
Y = torch.tensor([[1.0], [1.0], [1.0], [1.0], [0.0], [0.0], [0.0], [0.0]])

net = torch.nn.Sequential(
    torch.nn.Linear(4, 2),
    cuantica,
    torch.nn.Linear(cuantica.num_observables, 1),
)
print(net)

El entrenamiento es código normal de PyTorch: una pérdida de entropía cruzada binaria, el optimizador `torch.optim.Adam` sobre todos los parámetros de la red, incluidos los cuánticos, y 30 pasos. En cada paso de entrenamiento, el paso hacia delante llama una vez a `qml.expval` y el paso hacia atrás, una vez a `qml.vjp`:

In [ ]:
import matplotlib.pyplot as plt

loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.1)


def exactitud():
    with torch.no_grad():
        return ((net(X) > 0).float() == Y).float().mean().item()


print("exactitud antes:  ", exactitud())
perdidas = []
for paso in range(30):
    optimizer.zero_grad()
    loss = loss_fn(net(X), Y)
    loss.backward()
    optimizer.step()
    perdidas.append(loss.item())
print("exactitud después:", exactitud())

plt.figure(figsize=(5, 3))
plt.plot(perdidas)
plt.xlabel("paso")
plt.ylabel("entropía cruzada binaria")
plt.show()

## Partir de un modelo entrenado: `from_trained`

`from_trained` construye la capa a partir de un `TrainedModel`, el resultado de `fit` y `bind` o de `qml.load`, así que un modelo entrenado con Polypus, como en [`07a`](07a_clasificador_nativo.ipynb), puede seguir entrenándose dentro de una red de PyTorch. La capa nueva empieza en los parámetros entrenados, así que sus salidas coinciden con las de `qml.expval` para ellos:

In [ ]:
import numpy as np
import polypus
from polypus.torch import from_trained

x_cuatro = np.array([[0.2, 0.1], [0.4, 0.3], [2.7, 2.9], [3.0, 2.6]])
clasificador = qml.Model(
    qml.Encoder(2).angle_encoder("ry"),
    qml.Ansatz(2).real_amplitudes(1),
    qml.Readout([qml.Z(0)], decision="sign"),
)
result = clasificador.fit(
    x_cuatro,
    [1, 1, -1, -1],
    method=polypus.Adam(max_iters=30),
    loss="hinge",
    exact=True,
    seed=7,
)
entrenado = clasificador.bind(result.best_params)

capa = from_trained(entrenado, exact=True)
with torch.no_grad():
    print(capa(torch.tensor(x_cuatro)).squeeze(1))
print(qml.expval(clasificador, x_cuatro, entrenado.params, exact=True)[:, 0])

## Resumen

Este notebook ha convertido un modelo nativo en una capa de PyTorch con `QuantumLayer`, ha comprobado sus salidas y su gradiente exacto, la ha colocado entre dos capas `Linear` en una red híbrida entrenada con `torch.optim.Adam`, y, con `from_trained`, ha creado una capa a partir de un modelo entrenado con `fit`.

## Fin de la serie

Esta serie ha recorrido el camino completo: desde el primer circuito hasta entrenar modelos con datos. Quien quiera seguir puede consultar [`examples/`](https://github.com/Bahia-Software/polypus/tree/main/examples), con casos más grandes, entre ellos una versión completa de MaxCut con QAOA. Varios de ellos continúan esta sección: `qml_iris_demo.py` entrena un clasificador nativo con las flores Iris, y `qml_native_showcase.py` recorre todos los encoders, capas, optimizadores, lecturas y pérdidas nativos. `qml_expval_jacobian_loop.py` y `qml_torch_hybrid.py` son los scripts en los que se basan [`07c`](07c_expval_y_jacobian.ipynb) y este notebook. La [guía de usuario](https://github.com/Bahia-Software/polypus/tree/main/docs/guide), en inglés, trata a fondo el entrenamiento, QML, la memoria y el rendimiento; su [capítulo de QML](https://github.com/Bahia-Software/polypus/blob/main/docs/guide/qml.md) documenta `polypus.torch` por completo.

Volver al índice: [`00_bienvenida.ipynb`](00_bienvenida.ipynb).